# Session 11: Phylogenetics

**Module 3: Programming for Biological Data**  
**Date:** February 9, 2026 | 18:30 – 21:30  
**Instructor:** Dr. Haogao Gu

---

## Learning Objectives

By the end of this session, you will be able to:
1. Perform **Multiple Sequence Alignment (MSA)**
2. Calculate **genetic distance matrices**
3. Build **phylogenetic trees** (Neighbor-Joining)
4. Interpret trees for **outbreak investigation**

In [ ]:
# ============================================
# STANDARD SETUP PROTOCOL
# ============================================
options(repos = c(CRAN = "https://cloud.r-project.org"))

if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

# Install packages (may take a few minutes)
if (!requireNamespace("Biostrings", quietly = TRUE))
  BiocManager::install("Biostrings", update = FALSE, ask = FALSE)
if (!requireNamespace("msa", quietly = TRUE))
  BiocManager::install("msa", update = FALSE, ask = FALSE)
if (!requireNamespace("ape", quietly = TRUE))
  install.packages("ape")

library(Biostrings)
library(msa)
library(ape)

cat("✅ All packages loaded!")

---

# Part 1: Multiple Sequence Alignment

## 40 minutes

---

## 1.1 What is MSA?

**Multiple Sequence Alignment (MSA):**
- Arranging 3+ sequences to identify **homologous regions**
- Inserts **gaps** to maximize similarity
- Foundation for all phylogenetic analysis

## 1.2 Why Align?

| Without Alignment | With Alignment |
|-------------------|----------------|
| ATGCATGC | ATG-CATGC |
| ATGATGC | ATG-ATGC |
| ATGCATC | ATGCAT-C |

Alignment reveals where sequences **differ** (mutations)!

## 1.3 Popular Algorithms

| Algorithm | Speed | Accuracy | Best For |
|-----------|-------|----------|----------|
| **ClustalW** | Slow | Good | Small datasets |
| **MUSCLE** | Fast | Very Good | Medium datasets |
| **MAFFT** | Very Fast | Excellent | Large datasets |

## 1.4 Loading Sequences

In [ ]:
# Create demo sequences
seqs <- DNAStringSet(c(
  "Wuhan" = "ATGTTTGTTTTTCTTGTTTTATTGCCACTAGTCTCTAGTCAGTGTGTTAATCTTACAACC",
  "Alpha" = "ATGTTTGTTTTTCTTGTTTTATTGCCACTAGTCTAAAGTCAGTGTGTTAATCTTACAACC",
  "Delta" = "ATGTTTGTTTCACTTGTTTTATTGCCACTAGTCTCTAGTCAGTGTGTTAATCTCTCAACC",
  "Omicron" = "ATGTTTGTTTTTCTTGTAATTATTGCCACTAATCTCTAGTCAGTGTGTTAATCTTACAACC",
  "Patient1" = "ATGTTTGTTTTTCTTGTAATATTGCCACTAATCTCTAGTCAGTGTGTTCTTACAACT"
))

print(seqs)

## 1.5 Performing MSA with msa Package

In [ ]:
# Perform multiple sequence alignment
alignment <- msa(seqs, method = "ClustalW")

# View alignment
print(alignment, show = "complete")

In [ ]:
# View consensus sequence
consensus <- msaConsensusSequence(alignment)
cat("Consensus:", consensus)

## 1.6 Converting MSA for Tree Building

In [ ]:
# Convert MSA to DNAbin format (required by ape)
alignment_seqs <- as(alignment, "DNAStringSet")
dna_bin <- as.DNAbin(alignment_seqs)

cat("Converted to DNAbin format\n")
cat("Sequences:", length(dna_bin), "\n")
cat("Aligned length:", ncol(as.matrix(dna_bin)), "bp")

---

# Part 2: Tree Building (Phylogenetics)

## 40 minutes

---

## 2.1 What is a Phylogenetic Tree?

A **phylogenetic tree** shows evolutionary relationships:

- **Tips (leaves):** Individual sequences/samples
- **Nodes:** Common ancestors
- **Branches:** Evolutionary distance (mutations)
- **Root:** Most recent common ancestor

## 2.2 Genetic Distance

**Genetic distance** = how different are two sequences?

Simple measure: **p-distance** = proportion of different sites

$$d = \frac{\text{Number of differences}}{\text{Total sites compared}}$$

In [ ]:
# Calculate genetic distance matrix
dist_matrix <- dist.dna(dna_bin, model = "raw")  # p-distance

# View as matrix
print(round(as.matrix(dist_matrix), 4))

## 2.3 Distance Models

| Model | Description | Use Case |
|-------|-------------|----------|
| **raw** | Simple proportion | Quick look |
| **JC69** | Jukes-Cantor | Equal rates |
| **K80** | Kimura 2-parameter | Ts/Tv bias |
| **TN93** | Tamura-Nei | Most realistic |

## 2.4 Tree Building Methods

| Method | Speed | Accuracy | Algorithm |
|--------|-------|----------|----------|
| **Neighbor-Joining (NJ)** | Fast | Good | Distance-based |
| **UPGMA** | Very Fast | Fair | Distance-based |
| **Maximum Likelihood (ML)** | Slow | Excellent | Model-based |
| **Bayesian** | Very Slow | Excellent | Probabilistic |

## 2.5 Building a Neighbor-Joining Tree

In [ ]:
# Build Neighbor-Joining tree
tree <- nj(dist_matrix)

# View tree structure
print(tree)

In [ ]:
# Plot the tree
plot(tree, main = "Neighbor-Joining Tree: SARS-CoV-2 Variants")

# Add scale bar
add.scale.bar()

## 2.6 Tree Interpretation

**Reading a tree:**
- Closely clustered tips = recently diverged
- Long branches = more mutations
- Sister taxa = share common ancestor

**For outbreak investigation:**
- Patients clustering together = likely same transmission chain
- Patient clustering with known variant = identification

In [ ]:
# Root the tree (using Wuhan as outgroup)
rooted_tree <- root(tree, outgroup = "Wuhan", resolve.root = TRUE)

# Plot rooted tree
plot(rooted_tree, main = "Rooted Tree (Wuhan as outgroup)")
add.scale.bar()

## 2.7 Different Tree Visualizations

In [ ]:
# Different tree layouts
par(mfrow = c(2, 2))

plot(rooted_tree, type = "phylogram", main = "Phylogram")
plot(rooted_tree, type = "cladogram", main = "Cladogram")
plot(rooted_tree, type = "fan", main = "Fan/Radial")
plot(rooted_tree, type = "unrooted", main = "Unrooted")

par(mfrow = c(1, 1))

## 2.8 Saving Trees

In [ ]:
# Save tree in Newick format
write.tree(rooted_tree, file = "output_tree.nwk")
cat("Tree saved to output_tree.nwk\n")

# View Newick string
cat("\nNewick format:\n")
cat(write.tree(rooted_tree))

---

# Key Takeaways

1. **MSA** aligns sequences to identify mutations

2. **msa package:** `msa(seqs, method = "ClustalW")`

3. **Genetic distance:** How different are sequences

4. **ape package:** `dist.dna()` + `nj()` = tree

5. **Tree interpretation:** Clustering = related sequences

---

## Key Functions

```r
library(msa); library(ape)
alignment <- msa(seqs, method = "ClustalW")
dna_bin <- as.DNAbin(as(alignment, "DNAStringSet"))
dist_matrix <- dist.dna(dna_bin, model = "raw")
tree <- nj(dist_matrix)
plot(tree)
```

---

## Now proceed to Tutorial 11! 🌳